# Tutorial from Pyro

# Variational Autoencoder with a Normalizing Flow prior

Using a normalizing flow as prior for the latent variables instead of the typical standard Gaussian is an easy way to make a variational autoencoder (VAE) more expressive. This notebook demonstrates how to implement a VAE with a normalizing flow as prior for the MNIST dataset. We strongly recommend to read [Pyro's VAE tutorial](vae.ipynb) first.

> In this notebook we use [Zuko](https://zuko.readthedocs.io/) to implement normalizing flows, but similar results can be obtained with other PyTorch-based flow libraries.

In [ ]:
import pyro
import torch
import torch.nn as nn
import torch.utils.data as data
import zuko
import seaborn as sns
import numpy as np

from pyro.contrib.zuko import ZukoToPyro
from pyro.optim import Adam
from pyro.infer import SVI, Trace_ELBO
from torch import Tensor
from torchvision.datasets import MNIST
from torchvision.transforms.functional import to_tensor, to_pil_image
from tqdm import tqdm

## Data

The [MNIST](https://wikipedia.org/wiki/MNIST_database) dataset consists of 28 x 28 grayscale images representing handwritten digits (0 to 9).

In [ ]:
trainset = MNIST(root='', download=True, train=True, transform=to_tensor)
trainloader = data.DataLoader(trainset, batch_size=256, shuffle=True)

In [ ]:
x = [trainset[i][0] for i in range(16)]
x = torch.cat(x, dim=-1)

to_pil_image(x)

## Model

The encoder $q_\psi(z | x)$ is a (diagonal) Gaussian model, and the decoder $p_\phi(x | z)$ is a Bernoulli model. 

In [ ]:
class GaussianEncoder(nn.Module):
    def __init__(self, features: int, latent: int):
        super().__init__()

        self.hyper = nn.Sequential(
            nn.Linear(features, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 2 * latent),
        )

    def forward(self, x: Tensor):
        phi = self.hyper(x)
        mu, log_sigma = phi.chunk(2, dim=-1)

        return pyro.distributions.Normal(mu, log_sigma.exp()).to_event(1)


class BernoulliDecoder(nn.Module):
    def __init__(self, features: int, latent: int):
        super().__init__()

        self.hyper = nn.Sequential(
            nn.Linear(latent, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, features),
        )

    def forward(self, z: Tensor):
        phi = self.hyper(z)
        rho = torch.sigmoid(phi)

        return pyro.distributions.Bernoulli(rho).to_event(1)

We use a [masked autoregressive flow](https://arxiv.org/abs/1705.07057) (MAF) as prior $p_\phi(z)$ instead of the typical standard Gaussian $\mathcal{N}(0, I)$. Instead of implementing the MAF ourselves, we borrow it from the [Zuko](https://github.com/probabilists/zuko) library. Because Zuko distributions are very similar to Pyro distributions, a thin wrapper (`ZukoToPyro`) is sufficient to make Zuko and Pyro 100% compatible.

In [ ]:
class VAE(nn.Module):
    def __init__(self, features: int, latent: int = 16):
        super().__init__()

        self.encoder = GaussianEncoder(features, latent)
        self.decoder = BernoulliDecoder(features, latent)

        self.prior = zuko.flows.MAF(
            features=latent,
            transforms=3,
            hidden_features=(256, 256),
        )

    def model(self, x: Tensor):
        pyro.module("prior", self.prior)
        pyro.module("decoder", self.decoder)

        with pyro.plate("batch", len(x)):
            z = pyro.sample("z", ZukoToPyro(self.prior()))
            x = pyro.sample("x", self.decoder(z), obs=x)

    def guide(self, x: Tensor):
        pyro.module("encoder", self.encoder)

        with pyro.plate("batch", len(x)):
            z = pyro.sample("z", self.encoder(x))

vae = VAE(784, 16)
vae

## Training

We train our VAE with a standard stochastic variational inference (SVI) pipeline.

In [ ]:
pyro.clear_param_store()

svi = SVI(vae.model, vae.guide, Adam({'lr': 1e-3}), loss=Trace_ELBO())

for epoch in (bar := tqdm(range(96))):
    losses = []
    for x, _ in trainloader:
        x = x.round().flatten(-3)
        losses.append(svi.step(x))
    losses = torch.tensor(losses)

    bar.set_postfix(loss=losses.sum().item() / len(trainset))

After training, we can generate MNIST images by sampling latent variables from the prior and decoding them.

In [ ]:
vae.eval();

In [ ]:
z = vae.prior().sample((15,))
x = vae.decoder(z).mean.reshape(-1, 28, 28)

to_pil_image(x.movedim(0, 1).reshape(28, -1))

In [ ]:
sns.kdeplot(vae.prior().sample((1000,)).numpy())

In [ ]:
idx = 0
x_img, y = trainset[idx]

In [ ]:
to_pil_image(x_img)

In [ ]:
x = x_img.round().flatten().unsqueeze(0)

In [ ]:
with torch.no_grad():
    q_z_given_x = vae.encoder(x)        # q_psi(z | x)
    z_samples = q_z_given_x.sample((5000,))

# Shape: [num_samples, batch, latent]

z_samples.shape

In [ ]:
sns.kdeplot(z_samples.squeeze(1).numpy())

In [ ]:
S = 10_000

# Activity: importance sampling from the guide

For one fixed image $x$, use samples from the encoder

$$
z^{(s)} \sim q_\phi(z\,|\, x)
$$

to approximate the posterior

$$
p_\theta(z\,|\, x) \propto p_\theta(z)p_\theta(x \,|\, z).
$$

Complete the code below.

Questions:

1. Are the samples $z$ posterior samples?
2. What posterior object is approximated after adding the weights?
3. What does a small ESS mean here?

In [ ]:
with torch.no_grad():
    # 1. Build the guide distribution q_phi(z | x)
    # Hint: we need to use a method from "vae"
    q = ...
    # 2. Draw (S,) latent samples from the guide
    z = ...
    # 3. Evaluate the proposal log-density log q_phi(z | x)
    # hint: use .log_prob
    log_q = ...
    # 4. Evaluate the prior log-density log p_theta(z)
    # Hint1: We need to use a from "vae"
    # Hint2: We will have to use "ZukoToPyro"
    log_pz = ...
    # 5. Evaluate the log-likelihood log p_theta(x | z)
    # Hint: Again, a method from "vae"
    px_given_z = ...
    log_px_given_z = ...
    assert log_px_given_z.ndim == 2
    # 6. Compute log importance weights
    # log w = log p(z) + log p(x | z) - log q(z | x)
    log_w = ...
    log_w = log_w.squeeze(-1)
    # 7. Normalize weights
    # Hint: torch.softmax, `dim = 0`
    w = ...
    # 8. Effective sample size
    ess = 1.0 / torch.sum(w ** 2)

In [ ]:
print(f"ESS: {ess.item():.1f} / {S}")
print(f"max weight: {w.max().item():.4f}")
print(f"top 10 mass: {torch.topk(w, 10).values.sum().item():.4f}")
print(f"top 100 mass: {torch.topk(w, 100).values.sum().item():.4f}")

In [ ]:
sample = np.random.choice(len(z), size=len(z), replace=True, p=w.numpy())

In [ ]:
sns.kdeplot(z.squeeze(1).numpy()[sample])